# Validation #1: Classical SMC on Double Integrator

## Reference
**Liu, Jinkun (2017).** *Sliding Mode Control Using MATLAB.* Academic Press (Elsevier).
- Chapter 1, Section 1.1: "A Simple Sliding Mode Controller Design"
- Equations 1.1 -- 1.4, Figures 1.1 -- 1.6
- DOI: [10.1016/B978-0-12-802575-8.00001-1](http://dx.doi.org/10.1016/B978-0-12-802575-8.00001-1)

## Purpose
Reproduce the **exact** numerical example from Liu (2017) Ch 1 in Python,
then compare against OpenSMC MATLAB output. If results match the book's
figures, the implementation is validated.

## The Example (Liu, 2017, p. 3)

**Plant (Eq. 1.1):**
$$J\ddot{\theta}(t) = u(t) + d(t)$$
where $J = 2$ is the inertia, $d(t) = \sin t$ is the disturbance, $|d(t)| \leq \eta$.

**Sliding surface (Eq. 1.2):**
$$s(t) = c\, e(t) + \dot{e}(t), \quad c > 0$$
where $e = \theta - \theta_d$ (actual minus desired), $\dot{e} = \dot{\theta} - \dot{\theta}_d$.

**Control law (Eq. 1.4):**
$$u(t) = J\!\left(-c\,\dot{e} + \ddot{\theta}_d - \frac{1}{J}\!\left(k\,s + \eta\,\text{sgn}(s)\right)\right)$$

**Parameters:**
| Symbol | Value | Meaning |
|--------|-------|---------|
| $J$ | 2 | Inertia moment |
| $c$ | 10 | Surface slope |
| $\eta$ | 1.1 | Switching gain ($> \|d\|_\infty = 1$) |
| $k$ | 10 (Case 1) / 0 (Case 2) | Exponential reaching gain |
| $\theta_d$ | 1.0 | Step reference |
| $\theta(0), \dot\theta(0)$ | 0, 0 | Initial conditions |
| $T$ | 30 s | Simulation time |

**Validation targets (from book figures):**
- **Fig 1.1** (k=10): Position converges to 1.0 within ~2--3 s, no overshoot
- **Fig 1.2** (k=10): Control spike ~100 at t=0, then chattering ~$\pm$5
- **Fig 1.3** (k=10): Phase trajectory hits sliding line $\dot{e} = -ce$ and slides to origin
- **Fig 1.4** (k=0): Position converges slowly, ~15--20 s

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

# Reproduce figures in the book's style
plt.rcParams.update({
    'figure.figsize': (8, 5),
    'font.size': 12,
    'lines.linewidth': 1.5,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print("Libraries loaded.")

## Mathematical Derivation (following Liu, 2017, pp. 2--3)

### Step 1: Surface dynamics

Differentiate $s = ce + \dot{e}$:

$$\dot{s} = c\dot{e} + \ddot{e} = c\dot{e} + \ddot{\theta} - \ddot{\theta}_d = c\dot{e} + \frac{1}{J}(u + d) - \ddot{\theta}_d \tag{1.3}$$

### Step 2: Reaching condition

We need $s\dot{s} < 0$. Substituting the control law (1.4):

$$u = J\!\left(-c\dot{e} + \ddot{\theta}_d\right) - k\,s - \eta\,\text{sgn}(s)$$

into (1.3):

$$\dot{s} = c\dot{e} + \frac{1}{J}\!\Big[J(-c\dot{e} + \ddot{\theta}_d) - ks - \eta\,\text{sgn}(s) + d\Big] - \ddot{\theta}_d$$
$$= -\frac{k}{J}s - \frac{\eta}{J}\text{sgn}(s) + \frac{d}{J}$$

### Step 3: Lyapunov analysis

With $V = \frac{1}{2}s^2$:

$$\dot{V} = s\dot{s} = -\frac{k}{J}s^2 - \frac{\eta}{J}|s| + \frac{d}{J}s \leq -\frac{k}{J}s^2 - \frac{\eta - |d|}{J}|s|$$

Since $\eta = 1.1 > |d|_{\max} = 1$, we have $\dot{V} < 0$ for $s \neq 0$. $\square$

In [ ]:
# ============================================================
# EXACT REPRODUCTION of Liu (2017), Ch 1, Section 1.1
# ============================================================

# --- Parameters (from p. 3) ---
J     = 2.0       # Inertia moment
c     = 10.0      # Surface slope
eta   = 1.1       # Switching gain (must exceed |d|_max = 1)
theta_d = 1.0     # Step reference (position setpoint)

# Simulation
dt    = 1e-4      # Integration step (small enough for chattering resolution)
T     = 30.0      # Total time (matches book's figure x-axis)
N     = int(T / dt) + 1
t     = np.linspace(0, T, N)

# Initial conditions: theta(0) = 0, theta_dot(0) = 0
x0 = np.array([0.0, 0.0])  # [theta, theta_dot]


def disturbance(t):
    """d(t) = sin(t), as specified on p. 3."""
    return np.sin(t)


def reference(t):
    """Step reference: theta_d = 1.0, theta_d_dot = 0, theta_d_ddot = 0."""
    return theta_d, 0.0, 0.0


def plant_dynamics(t, x, u, d):
    """
    Plant: J * theta_ddot = u + d   (Eq. 1.1)
    State: x = [theta, theta_dot]
    """
    theta, theta_dot = x
    theta_ddot = (u + d) / J
    return np.array([theta_dot, theta_ddot])


def sliding_surface(e, edot):
    """s = c*e + edot   (Eq. 1.2)"""
    return c * e + edot


def control_law(e, edot, theta_d_ddot, s, k):
    """
    u = J * (-c*edot + theta_d_ddot - (1/J)*(k*s + eta*sgn(s)))   (Eq. 1.4)
    
    Expanded:
      u = J*(-c*edot + theta_d_ddot) - k*s - eta*sgn(s)
      u = u_eq + u_reaching
    where:
      u_eq      = J*(-c*edot + theta_d_ddot)   [equivalent control]
      u_reaching = -k*s - eta*sgn(s)           [reaching law]
    """
    u_eq = J * (-c * edot + theta_d_ddot)
    u_reach = -k * s - eta * np.sign(s)
    return u_eq + u_reach


def rk4_step(t, x, u, d, dt):
    """Standard 4th-order Runge-Kutta integration step."""
    k1 = plant_dynamics(t,        x,              u, d)
    k2 = plant_dynamics(t + dt/2, x + dt/2 * k1,  u, d)
    k3 = plant_dynamics(t + dt/2, x + dt/2 * k2,  u, d)
    k4 = plant_dynamics(t + dt,   x + dt   * k3,  u, d)
    return x + (dt / 6) * (k1 + 2*k2 + 2*k3 + k4)


def simulate(k_gain, dt=1e-4, T=30.0):
    """
    Full closed-loop simulation of Liu (2017) Example 1.1.
    
    Parameters:
        k_gain : exponential reaching gain (k=10 or k=0)
    
    Returns:
        dict with t, theta, theta_dot, e, edot, s, u, d arrays
    """
    N = int(T / dt) + 1
    t = np.linspace(0, T, N)
    
    # Storage
    theta     = np.zeros(N)
    theta_dot = np.zeros(N)
    e_arr     = np.zeros(N)
    edot_arr  = np.zeros(N)
    s_arr     = np.zeros(N)
    u_arr     = np.zeros(N)
    d_arr     = np.zeros(N)
    
    # Initial state
    x = x0.copy()
    
    for i in range(N):
        ti = t[i]
        
        # Reference
        td, td_dot, td_ddot = reference(ti)
        
        # Disturbance
        d = disturbance(ti)
        
        # Error (book convention: e = theta - theta_d)
        e = x[0] - td
        edot = x[1] - td_dot
        
        # Sliding variable
        s = sliding_surface(e, edot)
        
        # Control
        u = control_law(e, edot, td_ddot, s, k_gain)
        
        # Store
        theta[i]     = x[0]
        theta_dot[i] = x[1]
        e_arr[i]     = e
        edot_arr[i]  = edot
        s_arr[i]     = s
        u_arr[i]     = u
        d_arr[i]     = d
        
        # Integrate (skip last step)
        if i < N - 1:
            x = rk4_step(ti, x, u, d, dt)
    
    return {
        't': t, 'theta': theta, 'theta_dot': theta_dot,
        'e': e_arr, 'edot': edot_arr, 's': s_arr,
        'u': u_arr, 'd': d_arr, 'k': k_gain
    }


print("Simulation functions defined.")
print(f"Parameters: J={J}, c={c}, eta={eta}, theta_d={theta_d}")
print(f"Simulation: dt={dt}, T={T}s, N={N} steps")

## Case 1: k = 10 (Exponential + Switching Reaching Law)

This reproduces **Figures 1.1, 1.2, and 1.3** from the book.

In [ ]:
# Run simulation: k = 10
res_k10 = simulate(k_gain=10.0)

print(f"Case k=10 completed: {len(res_k10['t'])} steps")
print(f"  Final theta = {res_k10['theta'][-1]:.6f}  (target: {theta_d})")
print(f"  Final error = {res_k10['e'][-1]:.6f}")
print(f"  Max |u|     = {np.max(np.abs(res_k10['u'])):.2f}")
print(f"  Max |s|     = {np.max(np.abs(res_k10['s'])):.4f}")

In [ ]:
# --- Figure 1.1: Position Tracking (k=10) ---
fig, ax = plt.subplots()
ax.plot(res_k10['t'], res_k10['theta'], 'r-', label=r'$\theta(t)$')
ax.axhline(y=theta_d, color='k', linestyle='--', alpha=0.5, label=r'$\theta_d = 1.0$')
ax.set_xlabel('time (s)')
ax.set_ylabel('Step response')
ax.set_title('Figure 1.1 — Position tracking with k = 10  [Liu (2017)]')
ax.set_xlim([0, 30])
ax.set_ylim([0, 1.4])
ax.legend()
plt.tight_layout()
plt.savefig('fig_1_1_position_k10.png', dpi=150)
plt.show()

print("Compare with Liu (2017) Fig 1.1:")
print("  - Should converge to 1.0 within ~2-3 seconds")
print("  - No overshoot")
print("  - Smooth monotonic rise")

In [ ]:
# --- Figure 1.2: Control Input (k=10) ---
fig, ax = plt.subplots()
ax.plot(res_k10['t'], res_k10['u'], 'r-', linewidth=0.5)
ax.set_xlabel('time (s)')
ax.set_ylabel('Control input')
ax.set_title('Figure 1.2 — Control input with k = 10  [Liu (2017)]')
ax.set_xlim([0, 30])
ax.set_ylim([-20, 120])
plt.tight_layout()
plt.savefig('fig_1_2_control_k10.png', dpi=150)
plt.show()

print("Compare with Liu (2017) Fig 1.2:")
print("  - Initial spike ~100")
print("  - Chattering visible after convergence, amplitude ~+-5")
print(f"  Our max |u| = {np.max(np.abs(res_k10['u'])):.1f}")
print(f"  Our steady-state u range: [{np.min(res_k10['u'][-10000:]):.2f}, {np.max(res_k10['u'][-10000:]):.2f}]")

In [ ]:
# --- Figure 1.3: Phase Trajectory (k=10) ---
fig, ax = plt.subplots()
ax.plot(res_k10['e'], res_k10['edot'], 'k-', linewidth=0.8, label='Trajectory')

# Draw the sliding line: edot = -c*e
e_line = np.linspace(-1.1, 0.5, 100)
edot_line = -c * e_line
ax.plot(e_line, edot_line, 'r-', linewidth=2, label=f'Sliding line: $\dot{{e}} = -{c}e$')

ax.set_xlabel('e')
ax.set_ylabel('de')
ax.set_title('Figure 1.3 — Phase trajectory with k = 10  [Liu (2017)]')
ax.set_xlim([-1, 0.4])
ax.set_ylim([-2, 10])
ax.legend()
plt.tight_layout()
plt.savefig('fig_1_3_phase_k10.png', dpi=150)
plt.show()

print("Compare with Liu (2017) Fig 1.3:")
print("  - Trajectory starts at (e, edot) = (-1, 0)")
print("  - Hits the sliding line s=0 (red line)")
print("  - Then slides along it to the origin (0, 0)")

## Case 2: k = 0 (Pure Switching, No Exponential Term)

This reproduces **Figures 1.4--1.6** from the book. Without the $ks$ term,
reaching is driven only by $\eta\,\text{sgn}(s)$ — much slower.

In [ ]:
# Run simulation: k = 0
res_k0 = simulate(k_gain=0.0)

print(f"Case k=0 completed: {len(res_k0['t'])} steps")
print(f"  Final theta = {res_k0['theta'][-1]:.6f}  (target: {theta_d})")
print(f"  Final error = {res_k0['e'][-1]:.6f}")
print(f"  Max |u|     = {np.max(np.abs(res_k0['u'])):.2f}")

In [ ]:
# --- Figure 1.4: Position Tracking (k=0) ---
fig, ax = plt.subplots()
ax.plot(res_k0['t'], res_k0['theta'], 'k--', linewidth=1, label=r'$k = 0$')
ax.plot(res_k10['t'], res_k10['theta'], 'r-', linewidth=1, label=r'$k = 10$')
ax.axhline(y=theta_d, color='b', linestyle=':', alpha=0.5, label=r'$\theta_d = 1.0$')
ax.set_xlabel('time (s)')
ax.set_ylabel('Step response')
ax.set_title('Figure 1.4 — Position tracking: k=0 vs k=10  [Liu (2017)]')
ax.set_xlim([0, 30])
ax.set_ylim([0, 1.4])
ax.legend()
plt.tight_layout()
plt.savefig('fig_1_4_position_k0_vs_k10.png', dpi=150)
plt.show()

print("Compare with Liu (2017) Fig 1.4:")
print("  - k=0 converges much slower (~15-20 seconds)")
print("  - k=10 converges fast (~2-3 seconds)")
print("  - Both reach theta_d = 1.0 eventually")

## Quantitative Validation

Extract numerical metrics and compare against what's visible in the book's figures.

In [ ]:
def compute_metrics(res):
    """Extract quantitative metrics for validation."""
    t = res['t']
    theta = res['theta']
    u = res['u']
    s = res['s']
    e = res['e']
    
    # Settling time (2% band around theta_d)
    band = 0.02 * theta_d
    settled = np.abs(theta - theta_d) < band
    # Find first time it enters and stays in the band
    t_settle = None
    for i in range(len(t) - 1, -1, -1):
        if not settled[i]:
            if i < len(t) - 1:
                t_settle = t[i + 1]
            break
    if t_settle is None:
        t_settle = t[0]  # always in band
    
    # Reaching time: when |s| first drops below 0.1
    reaching_mask = np.abs(s) < 0.1
    t_reach = t[np.argmax(reaching_mask)] if np.any(reaching_mask) else T
    
    # Overshoot
    overshoot = (np.max(theta) - theta_d) / theta_d * 100  # percent
    
    # Steady-state error (last 1 second average)
    ss_idx = t > (T - 1.0)
    ss_error = np.mean(np.abs(e[ss_idx]))
    
    # Control effort
    max_u = np.max(np.abs(u))
    ss_u_range = (np.min(u[ss_idx]), np.max(u[ss_idx]))
    
    return {
        'settling_time_2pct': t_settle,
        'reaching_time': t_reach,
        'overshoot_pct': overshoot,
        'ss_error': ss_error,
        'max_control': max_u,
        'ss_control_range': ss_u_range,
        'final_theta': theta[-1],
    }


m_k10 = compute_metrics(res_k10)
m_k0  = compute_metrics(res_k0)

# ============================================================
# VALIDATION TABLE
# ============================================================
print("=" * 75)
print("  VALIDATION TABLE: Liu (2017) Ch 1, Section 1.1")
print("  Plant: J*theta_ddot = u + d,  J=2, d=sin(t)")
print("  Surface: s = 10*e + edot")
print("  Control: u = J(-c*edot + theta_d_ddot) - k*s - eta*sgn(s)")
print("=" * 75)
print()
print(f"{'Metric':<30} {'Book (k=10)':<20} {'Python (k=10)':<20} {'Python (k=0)':<20}")
print("-" * 90)
print(f"{'Settling time (2%)':<30} {'~2-3 s':<20} {m_k10['settling_time_2pct']:<20.3f} {m_k0['settling_time_2pct']:<20.3f}")
print(f"{'Reaching time (|s|<0.1)':<30} {'< 1 s (fast)':<20} {m_k10['reaching_time']:<20.4f} {m_k0['reaching_time']:<20.4f}")
print(f"{'Overshoot (%)':<30} {'~0% (none)':<20} {m_k10['overshoot_pct']:<20.3f} {m_k0['overshoot_pct']:<20.3f}")
print(f"{'Steady-state |e|':<30} {'~0':<20} {m_k10['ss_error']:<20.6f} {m_k0['ss_error']:<20.6f}")
print(f"{'Max |u|':<30} {'~100':<20} {m_k10['max_control']:<20.2f} {m_k0['max_control']:<20.2f}")
print(f"{'SS control range':<30} {'chatter +-5':<20} {'[{:.2f}, {:.2f}]'.format(*m_k10['ss_control_range']):<20} {'[{:.2f}, {:.2f}]'.format(*m_k0['ss_control_range']):<20}")
print(f"{'Final theta':<30} {'1.0':<20} {m_k10['final_theta']:<20.6f} {m_k0['final_theta']:<20.6f}")
print("-" * 90)
print()
print("MATCH ASSESSMENT:")
if m_k10['settling_time_2pct'] < 5.0 and m_k10['overshoot_pct'] < 1.0:
    print("  [PASS] k=10: Fast convergence, no overshoot — matches Fig 1.1")
else:
    print("  [FAIL] k=10: Does NOT match Fig 1.1!")

if m_k0['settling_time_2pct'] > 10.0:
    print("  [PASS] k=0:  Slow convergence — matches Fig 1.4")
else:
    print("  [FAIL] k=0:  Does NOT match Fig 1.4!")

if m_k10['max_control'] > 80 and m_k10['max_control'] < 150:
    print("  [PASS] k=10: Initial control spike ~100 — matches Fig 1.2")
else:
    print(f"  [WARN] k=10: Control spike = {m_k10['max_control']:.1f}, expected ~100")

## Sliding Variable Analysis

Verify the reaching and sliding phases explicitly.
- **Reaching phase:** $|s|$ decreases toward 0
- **Sliding phase:** $s \approx 0$, error decays as $e(t) = e(t_r)\exp(-c(t - t_r))$

In [ ]:
# --- Sliding Variable s(t) for both cases ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(res_k10['t'], res_k10['s'], 'r-', linewidth=0.5)
axes[0].set_xlabel('time (s)')
axes[0].set_ylabel('s(t)')
axes[0].set_title('Sliding variable s(t) — k = 10')
axes[0].set_xlim([0, 5])  # Zoom into reaching phase
axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.5)

axes[1].plot(res_k0['t'], res_k0['s'], 'b-', linewidth=0.5)
axes[1].set_xlabel('time (s)')
axes[1].set_ylabel('s(t)')
axes[1].set_title('Sliding variable s(t) — k = 0')
axes[1].set_xlim([0, 30])
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('fig_sliding_variable.png', dpi=150)
plt.show()

print(f"k=10: s(0) = {res_k10['s'][0]:.2f}, reaching time = {m_k10['reaching_time']:.4f} s")
print(f"k=0:  s(0) = {res_k0['s'][0]:.2f}, reaching time = {m_k0['reaching_time']:.4f} s")
print()
print("Theoretical s(0) = c*e(0) + edot(0) = 10*(-1) + 0 = -10")
print(f"Actual s(0) = {res_k10['s'][0]:.2f}  {'MATCH' if abs(res_k10['s'][0] - (-10)) < 0.01 else 'MISMATCH!'}")

## OpenSMC Gap Analysis

Comparing our Python implementation against what OpenSMC MATLAB currently does.

### Issue 1: Equivalent Control
Liu's control law (Eq. 1.4) has **equivalent control**:
$$u_{eq} = J(-c\dot{e} + \ddot{\theta}_d)$$

OpenSMC's `ClassicalSMC.m` line 53 has `u_eq = 0`. This is only correct
for J=1 and $\ddot{\theta}_d = 0$. For the Liu example (J=2), this would
produce **wrong results**.

### Issue 2: No Inertia Parameter
`DoubleIntegrator.m` has no inertia parameter J. The dynamics are
$\ddot{x} = u + d$ (implicitly J=1). To reproduce Liu's example, we need
either a J parameter or a general second-order plant.

### Issue 3: Error Sign Convention
- **Liu (2017):** $e = \theta - \theta_d$ (actual minus desired)
- **OpenSMC:** `e = xref - x` (desired minus actual)

Both are valid, but the control law signs flip. This must be documented
clearly in every validation.

### Issue 4: Reaching Law Decomposition
Liu's reaching term is $-ks - \eta\,\text{sgn}(s)$, which combines:
- `ExponentialRate` ($-ks - q\,\text{sgn}(s)$) when $k > 0$
- `ConstantRate` ($-\eta\,\text{sgn}(s)$) when $k = 0$

OpenSMC's reaching laws are correctly decomposed but the k=10 case
maps to `ExponentialRate('k', eta, 'q', k)` — note the parameter mapping.

## Conclusion

### Validation Status: Liu (2017), Ch 1, Section 1.1

| Check | Status |
|-------|--------|
| Python reproduces Fig 1.1 (position, k=10) | Run notebook to verify |
| Python reproduces Fig 1.2 (control, k=10) | Run notebook to verify |
| Python reproduces Fig 1.3 (phase, k=10) | Run notebook to verify |
| Python reproduces Fig 1.4 (position, k=0) | Run notebook to verify |
| s(0) = -10 (analytical check) | Run notebook to verify |
| Lyapunov Vdot < 0 verified | Proven analytically above |

### OpenSMC Action Items
1. **Add inertia J** to `DoubleIntegrator.m` (or create `SecondOrderSystem.m`)
2. **Fix `u_eq`** in `ClassicalSMC.m` — should compute equivalent control from plant model
3. **Add `ExponentialRate` mapping** documentation for Liu's combined reaching law
4. **Document sign convention** in README and all validation notebooks

### Citation for OpenSMC
```
Reference: Liu, J. (2017). Sliding Mode Control Using MATLAB.
           Academic Press. Chapter 1, Section 1.1, Eq. 1.1-1.4.
           DOI: 10.1016/B978-0-12-802575-8.00001-1
```